![](https://i.imgur.com/N0zUCi0.png)

# 📚 Exploring Document Loaders in LangChain

## Overview
This notebook provides a hands-on introduction to **Document Loaders** in LangChain—essential tools for importing data from various sources into your LLM applications. Document loaders transform raw files into structured `Document` objects that can be processed by LangChain.

## What You'll Learn
- **Understanding Document Loaders**: How they work and why they're crucial for building RAG (Retrieval-Augmented Generation) applications
- **Text File Loading**: Using `TextLoader` to import simple text files
- **PDF Processing**: Leveraging `PyMuPDFLoader` to extract content from PDF documents with page-level granularity
- **Word Document Handling**: Working with `Docx2txtLoader` to process Microsoft Word (.docx) files

## Key Concepts Covered
- The `Document` object structure (content + metadata)
- Different loader types for different file formats
- Inspecting loaded documents and their properties
- Understanding how documents are chunked and structured

## Prerequisites
- Basic Python knowledge
- Familiarity with LangChain fundamentals (recommended)

---

Let's dive in and explore how to bring external data into your LangChain workflows! 🚀

## Install Dependencies

In [ ]:
!pip install -q langchain==1.2.8
!pip install -q langchain-community==0.4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
!pip install -q pymupdf==1.25.3
!pip install -q docx2txt==0.9

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 79.7 MB/s eta 0:00:00


## Document Loaders

Document loaders are used to import data from various sources into LangChain as `Document` objects. A `Document` typically includes a piece of text along with its associated metadata.

### Examples of Document Loaders:

- **Text File Loader:** Loads data from a simple `.txt` file.
- **Web Page Loader:** Retrieves the text content from any web page.

### Functionality:

- **Load Method:** Each document loader has a `load` method that enables the loading of data as documents from a pre-configured source.
- **Lazy Load Option:** Some loaders also support a "lazy load" feature, which allows data to be loaded into memory gradually as needed.

For more detailed information, visit [LangChain's document loader documentation](https://docs.langchain.com/oss/javascript/langchain/knowledge-base#1-documents-and-document-loaders).


### Text Loader

The simplest loader reads in a file as text and places it all into one document.



In [ ]:
!curl -o README.md https://raw.githubusercontent.com/langchain-ai/langchain/master/README.md

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  6899  100  6899    0     0  47384      0 --:--:-- --:--:-- --:--:-- 47579


In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./README.md")
doc = loader.load()

In [ ]:
len(doc)

1

In [ ]:
type(doc[0])

langchain_core.documents.base.Document

In [ ]:
doc[0].metadata

{'source': './README.md'}

In [ ]:
print(doc[0].page_content[:3000])

<div align="center">
  <a href="https://www.langchain.com/">
    <picture>
      <source media="(prefers-color-scheme: light)" srcset=".github/images/logo-dark.svg">
      <source media="(prefers-color-scheme: dark)" srcset=".github/images/logo-light.svg">
      <img alt="LangChain Logo" src=".github/images/logo-dark.svg" width="80%">
    </picture>
  </a>
</div>

<div align="center">
  <h3>The platform for reliable agents.</h3>
</div>

<div align="center">
  <a href="https://opensource.org/licenses/MIT" target="_blank"><img src="https://img.shields.io/pypi/l/langchain" alt="PyPI - License"></a>
  <a href="https://pypistats.org/packages/langchain" target="_blank"><img src="https://img.shields.io/pepy/dt/langchain" alt="PyPI - Downloads"></a>
  <a href="https://pypi.org/project/langchain/#history" target="_blank"><img src="https://img.shields.io/pypi/v/langchain?label=%20" alt="Version"></a>
  <a href="https://vscode.dev/redirect?url=vscode://ms-vscode-remote.remote-containers/cloneInVo

### PDF Loaders

[Portable Document Format (PDF)](https://en.wikipedia.org/wiki/PDF), standardized as ISO 32000, is a file format developed by Adobe in 1992 to present documents, including text formatting and images, in a manner independent of application software, hardware, and operating systems.

LangChain integrates with a host of PDF parsers. Some are simple and relatively low-level; others will support OCR and image-processing, or perform advanced document layout analysis. The right choice will depend on your use-case and through experimentation.

Here we will see how to load PDF documents into the LangChain `Document` format

We download a research paper to experiment with

If the following command fails you can download the paper manually by going to http://arxiv.org/pdf/2103.15348.pdf, save it as `layoutparser_paper.pdf`and upload it on the left in Colab from the upload files option

In [ ]:
!gdown 1hPpUZnQWTAnhpmjnUJEOPE4hFwOrDIf6

Downloading...
From: https://drive.google.com/uc?id=1hPpUZnQWTAnhpmjnUJEOPE4hFwOrDIf6
To: /content/layoutparser_paper.pdf
100% 4.69M/4.69M [00:00<00:00, 45.2MB/s]


#### PyMuPDFLoader

This is the fastest of the PDF parsing options, and contains detailed metadata about the PDF and its pages, as well as returns one document per page. It uses the `pymupdf` library internally.

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("./layoutparser_paper.pdf")
pages = loader.load()

In [ ]:
len(pages)

16

In [ ]:
pages[0]

Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-06-22T01:27:10+00:00', 'source': './layoutparser_paper.pdf', 'file_path': './layoutparser_paper.pdf', 'total_pages': 16, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2021-06-22T01:27:10+00:00', 'trapped': '', 'modDate': 'D:20210622012710Z', 'creationDate': 'D:20210622012710Z', 'page': 0}, page_content='LayoutParser: A Uniﬁed Toolkit for Deep\nLearning Based Document Image Analysis\nZejiang Shen1 (\x00), Ruochen Zhang2, Melissa Dell3, Benjamin Charles Germain\nLee4, Jacob Carlson3, and Weining Li5\n1 Allen Institute for AI\nshannons@allenai.org\n2 Brown University\nruochen zhang@brown.edu\n3 Harvard University\n{melissadell,jacob carlson}@fas.harvard.edu\n4 University of Washington\nbcgl@cs.washington.edu\n5 University of Waterloo\nw422li@uwaterloo.ca\nAbstract. Recent advances in document image analysis (DIA) have been\nprimarily driven 

In [ ]:
pages[0].metadata

{'producer': 'pdfTeX-1.40.21',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2021-06-22T01:27:10+00:00',
 'source': './layoutparser_paper.pdf',
 'file_path': './layoutparser_paper.pdf',
 'total_pages': 16,
 'format': 'PDF 1.5',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2021-06-22T01:27:10+00:00',
 'trapped': '',
 'modDate': 'D:20210622012710Z',
 'creationDate': 'D:20210622012710Z',
 'page': 0}

In [ ]:
print(pages[0].page_content)

LayoutParser: A Uniﬁed Toolkit for Deep
Learning Based Document Image Analysis
Zejiang Shen1 ( ), Ruochen Zhang2, Melissa Dell3, Benjamin Charles Germain
Lee4, Jacob Carlson3, and Weining Li5
1 Allen Institute for AI
shannons@allenai.org
2 Brown University
ruochen zhang@brown.edu
3 Harvard University
{melissadell,jacob carlson}@fas.harvard.edu
4 University of Washington
bcgl@cs.washington.edu
5 University of Waterloo
w422li@uwaterloo.ca
Abstract. Recent advances in document image analysis (DIA) have been
primarily driven by the application of neural networks. Ideally, research
outcomes could be easily deployed in production and extended for further
investigation. However, various factors like loosely organized codebases
and sophisticated model conﬁgurations complicate the easy reuse of im-
portant innovations by a wide audience. Though there have been on-going
eﬀorts to improve reusability and simplify deep learning (DL) model
development in disciplines like natural language processing

In [ ]:
print(pages[4].page_content)

LayoutParser: A Uniﬁed Toolkit for DL-Based DIA
5
Table 1: Current layout detection models in the LayoutParser model zoo
Dataset
Base Model1 Large Model
Notes
PubLayNet [38]
F / M
M
Layouts of modern scientiﬁc documents
PRImA [3]
M
-
Layouts of scanned modern magazines and scientiﬁc reports
Newspaper [17]
F
-
Layouts of scanned US newspapers from the 20th century
TableBank [18]
F
F
Table region on modern scientiﬁc and business document
HJDataset [31]
F / M
-
Layouts of history Japanese documents
1 For each dataset, we train several models of diﬀerent sizes for diﬀerent needs (the trade-oﬀbetween accuracy
vs. computational cost). For “base model” and “large model”, we refer to using the ResNet 50 or ResNet 101
backbones [13], respectively. One can train models of diﬀerent architectures, like Faster R-CNN [28] (F) and Mask
R-CNN [12] (M). For example, an F in the Large Model column indicates it has a Faster R-CNN model trained
using the ResNet 101 backbone. The platform is maintained and

### Microsoft Office Document Loaders

The Microsoft Office suite of productivity software includes Microsoft Word, Microsoft Excel, Microsoft PowerPoint, Microsoft Outlook, and Microsoft OneNote. It is available for Microsoft Windows and macOS operating systems. It is also available on Android and iOS.

We leverage the [doc2txt](https://github.com/ankushshah89/python-docx2txt) package to process word documents.

Here we will leverage LangChain's [`Docx2txtLoader`](https://docs.langchain.com/oss/python/integrations/document_loaders/microsoft_word#using-docx2txt) to load data from a MS Word document.

In [ ]:
!gdown 1DEz13a7k4yX9yFrWaz3QJqHdfecFYRV-

Load word doc with complex parsing and section based chunks

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader("./Quantum Computing.docx")

data = loader.load()

In [ ]:
len(data)

1

In [ ]:
print(data[0].page_content)

The Rise of Quantum Computing: A New Era of Innovation

For decades, classical computing has driven technological advancements, but the limitations of traditional binary processing are becoming evident as the world demands more computational power. Enter quantum computing—a revolutionary approach that leverages the principles of quantum mechanics to solve complex problems at unprecedented speeds.

Understanding Quantum Computing

Unlike classical computers that process information using bits (0s and 1s), quantum computers use qubits, which can exist in multiple states simultaneously due to superposition. This unique property allows quantum systems to process vast amounts of data in parallel, making them exponentially more powerful for specific tasks.

Another key principle, entanglement, enables qubits to be interconnected, meaning the state of one qubit is dependent on another, regardless of distance. This drastically enhances processing efficiency and speed, paving the way for breakt